# Notebook 03 — VLM Baseline Inference

**Objective.** Standardized, deterministic inference through the common
interface; compare standard vs grounded prompting on several real images,
across both real E1 models — a lightweight sanity check of the adapters
before the full E1 pipeline (notebook 04).

**Inputs.** A handful of real GQA images (one per bucket, up to
`N_EXAMPLES`) + both real E1 models (`qwen3_vl_8b_ollama`, `llava_ollama`)
via Ollama — see `notes/OLLAMA.md`.

**Expected outputs.** Raw outputs saved per model/image/condition, plus a
side-by-side (standard vs grounded) preview alongside each image.

_Companion notebook: runnable top-to-bottom, relative paths, no hidden state, saves artifacts to `results/`._

In [ ]:
import sys, os
# Reusable logic lives in src/grounded_memory/. Notebooks call it.
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from pathlib import Path
RESULTS = Path('..') / 'results'
from grounded_memory import __version__
print('grounded_memory', __version__)

## Select a few sample images

Pick one real image from each of the first `N_EXAMPLES` buckets in the frozen manifest — a lightweight multi-image check (not the full 6-bucket preview; see notebook 04 for that).

In [ ]:
import csv

from grounded_memory.data.selection import BUCKETS

manifest_path = Path('..') / 'data' / 'manifests' / 'gqa_benchmark_v1.csv'
with manifest_path.open(encoding='utf-8') as fh:
    manifest_rows = list(csv.DictReader(fh))

by_bucket = {}
for row in manifest_rows:
    by_bucket.setdefault(row['primary_bucket'], row)

N_EXAMPLES = 4
sample_rows = [by_bucket[b] for b in list(BUCKETS)[:N_EXAMPLES]]
for row in sample_rows:
    print(f"{row['primary_bucket']:<20} image_id={row['image_id']}")

## Load the real models (Ollama)

Both E1 models: `qwen3_vl_8b_ollama` (primary baseline) and `llava_ollama` (family-contrast, same ~7–8B scale — see `configs/models.yaml`). Requires a local Ollama server with both models pulled (`ollama pull qwen3-vl:8b && ollama pull llava:7b`) — see `notes/OLLAMA.md`. Constructing the adapters never touches the network; connection errors only surface per-call, in the run cell below.

In [ ]:
from grounded_memory.config import load_config
from grounded_memory.models import load_model

MODEL_NAMES = ['qwen3_vl_8b_ollama', 'llava_ollama']  # both real E1 models
models_cfg = load_config('models')
models = {name: load_model(name, models_cfg) for name in MODEL_NAMES}
for name, m in models.items():
    print(f'Loaded {name} -> {m.model_id}')

In [ ]:
# gqa.root is repo-root-relative -- resolve it against the repo root explicitly,
# since the notebook kernel's cwd is notebooks/, not the repo root (same fix as
# notebook 02 / notebook 04).
data_cfg = load_config('datasets')
REPO_ROOT = Path('..').resolve()
gqa_root = Path(data_cfg['gqa']['root'])
if not gqa_root.is_absolute():
    gqa_root = (REPO_ROOT / gqa_root).resolve()
GQA_IMAGES_DIR = gqa_root / data_cfg['gqa']['images_dir']
print(f'Images dir: {GQA_IMAGES_DIR} (exists: {GQA_IMAGES_DIR.exists()})')

## Deterministic inference: standard vs grounded, both models

Run with temperature=0 (default). For each sample image, preview it once, then run both `standard` and `grounded` conditions against both real models — no mocking. If a model's Ollama endpoint is unreachable, that model is skipped for the rest of the run (not the whole notebook) and a warning is printed once.

In [ ]:
import textwrap

import matplotlib.pyplot as plt
from PIL import Image

from grounded_memory.pipelines.scene_understanding import run_conditions_for_image
from grounded_memory.tracking import make_run_id

run_id = make_run_id('nb03-baseline')
all_records = []
failed_models = set()

for row in sample_rows:
    image_id = row['image_id']
    image_path = GQA_IMAGES_DIR / f'{image_id}.jpg'

    fig, ax = plt.subplots(figsize=(4, 4))
    if image_path.exists():
        ax.imshow(Image.open(image_path))
    else:
        ax.text(0.5, 0.5, f'image not found:\n{image_path}', ha='center', va='center', wrap=True)
    ax.set_axis_off()
    ax.set_title(f"{row['primary_bucket']}  {image_id}")
    plt.show()

    for name in MODEL_NAMES:
        if name in failed_models:
            continue
        model = models[name]

        try:
            recs = run_conditions_for_image(
                model, str(image_path), sample_id=image_id,
                experiment_id='E1', run_id=run_id, conditions=['standard', 'grounded'],
                dataset='gqa', dataset_version=data_cfg['manifest']['version'],
                manifest_version=data_cfg['manifest']['version'],
            )
        except RuntimeError as e:
            print(f'\n  [skipping {name} for the rest of this run] {e}\n'
                  f'  Start Ollama (`ollama serve`) and pull the model, then re-run this cell. '
                  f'See notes/OLLAMA.md.')
            failed_models.add(name)
            continue

        print(f'\n  -- {name} ({model.model_id}) --')
        by_condition = {r.metrics['condition']: r for r in recs}
        for condition in ('standard', 'grounded'):
            r = by_condition[condition]
            text = str(r.parsed_prediction) if r.parse_ok else r.prediction
            preview = textwrap.shorten(text.replace('\n', ' '), width=300, placeholder=' …')
            print(f'    [{condition:<8}] parse_ok={r.parse_ok}  latency_s={r.latency_s:.2f}\n      {preview}')
        all_records.extend(recs)

print(f'\n{len(all_records)} predictions collected across {len(sample_rows)} images x {len(MODEL_NAMES)} models x 2 conditions.')

In [ ]:
from grounded_memory.tracking import write_jsonl

# Distinct prefix + fresh timestamped run_id -- never collides with, or
# overwrites, the frozen e1_*.jsonl files from the real 720-prediction run.
preview_path = RESULTS / 'predictions' / f'nb03_baseline_preview__{run_id}.jsonl'
if all_records:
    write_jsonl(all_records, preview_path, overwrite=True)
    print(f'wrote {len(all_records)} predictions -> {preview_path}')
else:
    print('No records collected (see error above) -- nothing written.')

## Findings

_Fill in 3–5 concrete findings after running this notebook._

1. …
2. …
3. …

> Do not populate tables with invented numbers. If the full model/data is unavailable, leave clearly-labeled placeholders.